# 7 . Declarative rule-builder

A strategy is a **list of tagged indicators** plus a **list of entry→exit rules**. Nothing is
defined in this notebook — every helper lives in `quant`, so the flow stays a clean sequence:

> **1** data → **2** indicators (tags) → **3** rules → **4** config + run → **5** trades → **6** chart

### Tagged indicators
You give each indicator a **tag**, and the tag *is* its column name. The same indicator with
different settings therefore never collides (`RSI('rsi_f', 7)` vs `RSI('rsi_s', 21)`), and rules
reference the **handle** — not a fragile string:

```python
trend = MA('trend', 200, type='sma', smooth=9, smooth_type='ema')   # -> column 'trend'
entry = full_above(trend, n=2)                                       # handle, not 'ma_200'
```

`MA(...)` maps 1:1 onto TradingView's MA dialog — Length, base `type`, **Smoothing Line**
(`smooth_type`) and **Smoothing Length** (`smooth`). Add `tf='1h'` to any indicator for a
higher-timeframe version (merged anti-lookahead). Declaring one tag twice with different
settings raises an error instead of silently overwriting.

### This strategy
- **`breakout`** (primary) — one entry per crossing: a candle forms fully above the trend MA;
  re-arms only when a candle fully forms *below* it. Exit: full candle below the fast MA **or**
  3 consecutive Heikin-Ashi red candles. Stop: previous swing low.
- **`pullback`** (secondary) — only *after* the breakout has traded & closed this episode.

> Restart the kernel after editing library code.

## 0 . Setup

In [12]:
import sys, os, warnings; warnings.filterwarnings('ignore')
try:
    import quant
except ModuleNotFoundError:
    sys.path.insert(0, os.path.abspath('..')); import quant
import pandas as pd
from quant.data import get_ohlcv
from quant.engine import BacktestConfig
from quant.reporting import print_summary
from quant.viz import ResearchChart, heikin_ashi_candles, open_in_browser
from quant.strategies import RuleStrategy, Rule, MA, RSI, HeikinAshi, Swings
from quant.strategies.conditions import (
    above, below, full_above, full_below, red, green, big_candle,
    close_of, ref_col_stop, pct_stop)
pd.set_option('display.max_rows', 200)
print('quant', quant.__version__)

quant 0.1.0


## 1 . Data
Swap to `('XAUUSD', 'dukascopy', 'cfd')` for true spot gold. Keep the window long enough to warm
up your slowest indicator (a 1h EMA-50 needs ~50 hours of data, or it stays NaN and never fires —
`prepare()` warns you if that happens).

In [22]:
SYMBOL, SOURCE, MARKET = 'XAUUSD', 'dukascopy', 'cfd'
START, END = '2026-07-21', '2026-07-22'
df = get_ohlcv(SYMBOL, '1m', start=START, end=END, source=SOURCE, market=MARKET, tz='UTC')
print(f'{len(df):,} 1m bars  {df.open_time.min()} -> {df.open_time.max()}')

[08:38:54] INFO | quant.data | cache HIT  dukascopy XAUUSD 1m -> pushdown load


1,381 1m bars  2026-07-21 00:00:00+00:00 -> 2026-07-22 00:00:00+00:00


## 2 . Indicators — declare once, tag them
The tag *is* the column name, so the same indicator with different settings never collides.

`MA_SOURCE` picks what the moving averages are computed on: `'close'` (real price) or `ha.close`
(Heikin Ashi). This is **separate** from the per-condition candle choice in section 3 — you can run
MAs on real price while testing candle conditions on HA, or vice-versa.

In [23]:
ha = HeikinAshi('ha')                       # ha.open/.high/.low/.close
sw = Swings('sw', 5, 5)                     # sw.last_low / sw.last_high (structure stop source)

MA_SOURCE = 'close'                         # 'close' = real price   |   ha.close = Heikin Ashi
trend = MA('trend', 200, type='sma', smooth=9, smooth_type='ema', source=MA_SOURCE)
fast  = MA('fast',   20, type='sma', smooth=9, smooth_type='ema', source=MA_SOURCE)

INDICATORS = [ha, sw, trend, fast]          # ha first: the MAs may read ha_close
print(f'MA source: {MA_SOURCE}')
for i in INDICATORS:
    print(f'  {i.tag:6s} -> {", ".join(i.columns)}')

MA source: close
  ha     -> ha_open, ha_high, ha_low, ha_close, ha_green
  sw     -> sw_high, sw_low, sw_last_high, sw_last_low
  trend  -> trend
  fast   -> fast


## 3 . Rules — one settings table, then the rules
Every condition picks its **own** candle type and confirmation length:

| setting | meaning |
|---|---|
| `*_SRC` | `'real'` = normal OHLC · `ha` = Heikin Ashi |
| `*_N` | how many consecutive candles must confirm |

So you can (say) enter on real candles but exit on 3 Heikin-Ashi reds, or flip any one of them
independently. Change the table, re-run — the rules, the printed summary and the chart all follow.

`once=True` = one entry per episode (re-arms on `reset`). `after='breakout'` = eligible only once
the breakout fired this episode. Conditions compose with `&` `|` `~`.

In [24]:
# ── Per-condition settings ──  SRC: 'real' = normal OHLC | ha = Heikin Ashi  ·  N: confirm bars
RESET_SRC,    RESET_N    = ha, 1     # full candle BELOW trend MA  -> ends the episode
BREAKOUT_SRC, BREAKOUT_N = ha, 1     # full candle ABOVE trend MA  -> primary entry
PULLBACK_SRC, PULLBACK_N = ha, 2     # full candle ABOVE fast MA   -> secondary entry
EXIT_MA_SRC,  EXIT_MA_N  = ha, 1     # full candle BELOW fast MA   -> exit
EXIT_RED_SRC, EXIT_RED_N = ha, 3     # N consecutive RED candles   -> exit

# ── Conditions built from the table above ──────────────────────────────────────────────────────
reset_c  = full_below(trend, RESET_N,    src=RESET_SRC)
entry_bo = full_above(trend, BREAKOUT_N, src=BREAKOUT_SRC)
entry_pb = above(close_of(PULLBACK_SRC), trend) & full_above(fast, PULLBACK_N, src=PULLBACK_SRC)
EXIT     = full_below(fast, EXIT_MA_N, src=EXIT_MA_SRC) | red(EXIT_RED_N, src=EXIT_RED_SRC)
STOP     = ref_col_stop(sw.last_low, fallback_pct=1.0)   # stop stays on REAL price levels

strat = RuleStrategy(
    indicators=INDICATORS,
    reset=reset_c,
    rules=[
        Rule('breakout', entry=entry_bo, exit=EXIT, stop=STOP, once=True),
        Rule('pullback', entry=entry_pb, exit=EXIT, stop=STOP, after='breakout', once=True),
    ],
)

print('condition        candles   N')
for lbl, s, n in [('reset', RESET_SRC, RESET_N), ('breakout entry', BREAKOUT_SRC, BREAKOUT_N),
                  ('pullback entry', PULLBACK_SRC, PULLBACK_N), ('exit: below MA', EXIT_MA_SRC, EXIT_MA_N),
                  ('exit: red run', EXIT_RED_SRC, EXIT_RED_N)]:
    print(f'  {lbl:14s} {"HA" if s is ha else "real":6s}  {n}')
print()
print(strat.describe())

condition        candles   N
  reset          HA      1
  breakout entry HA      1
  pullback entry HA      2
  exit: below MA HA      1
  exit: red run  HA      3

RuleStrategy 'rule_strategy' (2 rules):
  [long] breakout once: entry hafull_above(trend,1)  ->  exit (hafull_below(fast,1) | hared(3))
  [long] pullback after=breakout once: entry (ha_close>trend & hafull_above(fast,2))  ->  exit (hafull_below(fast,1) | hared(3))
  reset (episode boundary): hafull_below(trend,1)


## 4 . Config + backtest
Costs/sizing live on `BacktestConfig`; stops come from the rules above (`sl_mode='none'` here).

In [25]:
cfg = BacktestConfig(
    initial_cash=10_000, spread=0.12, slippage_bps=1.0, exit_enabled=True,
    sl_mode='none', tp_mode='none',              # per-rule stops are set on the Rules
    sizing_mode='risk_pct_equity', sizing_value=1.0, allow_rule_close=True,
)
res = strat.backtest(df, cfg)
print_summary(res, df=df)

[08:38:55] INFO | quant.engine | backtest done | bars=1,381 trades=5 final_cash=10007.77 return=0.08% dd=0.18% | 0.3 ms


 PERFORMANCE SUMMARY
  Total return %                       0.08
  Final cash                      10,007.77
  Trades                                  5
  Win rate %                          40.00
  Profit factor                        1.64
  Expectancy / trade                 1.5532
  Sharpe                               5.71
  Sortino                              2.13
  Calmar                             169.04
  Max drawdown %                       0.18
  Recovery factor                      0.44
  Avg bars held                          11
  Total fees                           0.00

  Best weekday: Tue (pnl=7.77, win=40.0%)
  Best session: newyork (pnl=17.87, n=2)


## 5 . Trades

In [26]:
cols = ['side','entry_time','entry_price','stop_price','exit_time','exit_price',
        'pnl','return_pct','bars_held','close_reason']
t = res.trades[cols].round({'pnl':2,'return_pct':2,'entry_price':3,'exit_price':3,'stop_price':3})
print('close-reason counts:'); display(t['close_reason'].value_counts())
t

close-reason counts:


close_reason
signal    5
Name: count, dtype: int64

,side,entry_time,entry_price,stop_price,exit_time,exit_price,pnl,return_pct,bars_held,close_reason
0,long,2026-07-21 03:27:00+00:00,4046.410,4036.295,2026-07-21 03:33:00+00:00,4043.921,-6.15,-0.06,6,signal
1,long,2026-07-21 10:22:00+00:00,4067.812,4060.525,2026-07-21 10:29:00+00:00,4066.068,-4.28,-0.04,7,signal
2,long,2026-07-21 14:12:00+00:00,4062.721,4044.165,2026-07-21 14:22:00+00:00,4062.039,-1.68,-0.02,10,signal
3,long,2026-07-21 18:21:00+00:00,4073.842,4070.965,2026-07-21 18:31:00+00:00,4081.817,19.55,0.20,10,signal
4,long,2026-07-21 22:23:00+00:00,4082.063,4076.425,2026-07-21 22:46:00+00:00,4082.197,0.33,0.00,23,signal


## 6 . Chart
Charts the **prepared** frame and plots the indicator handles, so every line drawn is literally the
column the rules evaluated. Opens in a browser tab with a Real / Heikin-Ashi candle toggle.

In [27]:
prep = strat.prepare(df)
ch = ResearchChart(prep, candles=True, initial_bars=len(prep), candle_target=len(prep),
                   title=f'{SYMBOL} 1m - rule-builder', candle_name='Real candles')
ch.add_indicator(trend, fast)                      # reads each handle's own column
ch.add_candle_alt(heikin_ashi_candles(df), name='Heikin Ashi')
ch.add_trades(res.trades)
print('chart ->', open_in_browser(ch.show(), 'rule_builder_chart.html'))

chart -> C:\Users\Talal Zahid\Desktop\Talal\Binance\notebooks\rule_builder_chart.html
